# PEA Studio · Free open-source video engine

This notebook runs the official **Wan2.1 T2V-1.3B** model using a free Kaggle GPU session when one is available. It downloads the model from **ModelScope** (not Hugging Face), uses the official Wan source code from GitHub, and exports an MP4 you can import into PEA Studio.

**Starting profile:** 480p, about 2.5 seconds, 25 sampling steps. This is a test-friendly starting point; increase frames/steps after the first successful render. The official project lists 8.19 GB of VRAM for T2V-1.3B. GPU availability, speed and session time are controlled by Kaggle and are not guaranteed.

**Model licence:** Apache 2.0. Read the model card and licence before commercial use.

## Run it
1. In Kaggle, create a new notebook and import this `.ipynb` file.
2. In notebook settings, turn Internet on and choose a GPU accelerator (T4x2 if offered).
3. Run the cells from top to bottom. The model download is large (about 17.6 GB on ModelScope), so keep enough session storage available.
4. Edit the prompt in the **Create your clip** cell and run it.
5. Download the resulting MP4 from the notebook output and upload it into PEA Studio using **Import finished clip**.

This is a free-compute route, not an always-on hosted API: Kaggle may limit or decline GPU sessions, and you must bring each finished MP4 back into PEA.


In [ ]:
# 1 · Check the actual notebook hardware before downloading the model.
import os, sys, subprocess, torch

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Kaggle notebook Settings, select a GPU accelerator, then rerun this cell.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} · {p.total_memory / 1024**3:.1f} GB VRAM')
if torch.cuda.get_device_properties(0).total_memory < 8 * 1024**3:
    raise RuntimeError('The first GPU has less than 8 GB VRAM. This starter profile is not suitable for it.')
print('Disk space:')
subprocess.run(['bash', '-lc', 'df -h /kaggle/working /kaggle/temp 2>/dev/null || df -h'], check=False)


## 2 · Install the open-source engine

This uses the official `Wan-Video/Wan2.1` GitHub repository. FlashAttention is optional in the official attention module; when it is not installed, the code falls back to PyTorch scaled-dot-product attention, which can be slower.


In [ ]:
from pathlib import Path
import subprocess, sys

WORK = Path('/kaggle/working')
REPO = WORK / 'Wan2.1'
MODEL_DIR = Path('/kaggle/temp/Wan2.1-T2V-1.3B')
OUT_DIR = WORK / 'PEA-Exports'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not (REPO / 'generate.py').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Wan-Video/Wan2.1.git', str(REPO)], check=True)

# Keep Kaggle's preinstalled CUDA-enabled torch/torchvision.
packages = [
    'numpy>=1.23.5,<2', 'opencv-python>=4.9.0.80', 'diffusers>=0.31.0,<1',
    'transformers>=4.49.0,<5', 'tokenizers>=0.20.3,<1', 'accelerate>=1.1.1,<2',
    'tqdm', 'imageio', 'easydict', 'ftfy', 'dashscope', 'imageio-ffmpeg',
    'modelscope'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Wan source and Python dependencies are ready.')


## 3 · Download the model weights from ModelScope

The official ModelScope model card reports a download size of about 17.57 GB. This step is needed only once per fresh Kaggle session because notebook disks are temporary.


In [ ]:
from modelscope.hub.snapshot_download import snapshot_download

# The SDK downloader resumes interrupted files and reuses completed files.
snapshot_download('Wan-AI/Wan2.1-T2V-1.3B', local_dir=str(MODEL_DIR))

weight_files = list(MODEL_DIR.rglob('*.safetensors'))
print(f'Found {len(weight_files)} safetensors files in {MODEL_DIR}')
if not weight_files:
    raise RuntimeError('ModelScope did not leave model weight files in the expected folder. Check the download output and available disk space.')


## 4 · Create your clip

Change `PROMPT` for each shot. For a continuing film, reuse the same character description in every prompt. Text-to-video does not guarantee identical characters across separate clips, so treat each result as a shot to review rather than assuming perfect continuity.


In [ ]:
import subprocess, sys, time

PROMPT = (
    'A beautifully animated cinematic scene: a young Kenyan girl with short natural hair, '
    'wearing a mustard-yellow cardigan and a small red backpack, walks along a sunlit Nairobi street. '
    'She pauses as a tiny paper bird flutters past her face. Gentle character animation, '
    'expressive movement, warm afternoon light, smooth camera tracking, detailed hand-painted '
    '3D animated-film look, coherent anatomy, stable background, no subtitles, no text, no logos.'
)

output_file = OUT_DIR / f'pea_wan21_{time.strftime("%Y%m%d_%H%M%S")}.mp4'
command = [
    sys.executable, str(REPO / 'generate.py'),
    '--task', 't2v-1.3B',
    '--size', '832*480',
    '--ckpt_dir', str(MODEL_DIR),
    '--prompt', PROMPT,
    '--offload_model', 'True',
    '--t5_cpu',
    '--sample_shift', '8',
    '--sample_guide_scale', '6',
    '--sample_steps', '25',
    '--frame_num', '41',
    '--save_file', str(output_file),
]
print('Starting Wan2.1. The first render can take a while on a free notebook GPU.')
subprocess.run(command, cwd=str(REPO), check=True)
if not output_file.exists() or output_file.stat().st_size == 0:
    raise RuntimeError('The generation command finished without a usable MP4 output.')
print(f'Finished: {output_file} · {output_file.stat().st_size / 1024**2:.1f} MB')


In [ ]:
# 5 · Preview and download the MP4, then import it into PEA Studio.
from IPython.display import Video, FileLink, display
display(Video(str(output_file), embed=False))
display(FileLink(str(output_file)))
print('In PEA Studio, open Create with AI → Import finished clip, then select this MP4.')
